In [ ]:
# ============================================================
# 02_keypoint_extraction_colab
# Cloud-Based Intelligent Taekwondo Motion Classification System
#
# Purpose:
# 1. Install and load MediaPipe Pose Landmarker
# 2. Extract pose keypoints from taekwondo videos
# 3. Select the main actor when two people appear
# 4. Test / inspect pose candidates visually
# 5. Batch process videos from taekwondo_manifest_clean.csv
# 6. Save keypoint CSV files to Google Drive
# 7. Save error log for failed videos
# ============================================================


# ============================================================
# 0. Safe Google Drive Mount
# ============================================================

import os

if os.path.exists("/content/drive/MyDrive"):
    print("Google Drive is already mounted.")
else:
    from google.colab import drive
    drive.mount("/content/drive")


# ============================================================
# 1. Install Required Packages
# ============================================================

# Run this once in Colab. If Colab asks to restart runtime after installation,
# restart runtime and rerun from the top.
!pip -q install mediapipe opencv-python pandas numpy matplotlib tqdm


# ============================================================
# 2. Imports and Global Paths
# ============================================================

import os
import cv2
import json
import traceback
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mediapipe as mp

from pathlib import Path
from tqdm.notebook import tqdm


ROOT = "/content/drive/MyDrive"

# Input manifest generated by 01_data_processing_colab
MANIFEST_PATH = f"{ROOT}/taekwondo_manifest_clean.csv"

# Output folder for extracted keypoint CSV files
OUTPUT_DIR = f"{ROOT}/taekwondo_keypoints_csv_main_actor"

# Error log
ERROR_LOG_PATH = f"{ROOT}/taekwondo_keypoints_error_log.csv"

# MediaPipe model path
MODEL_DIR = "/content/mediapipe_models"
MODEL_PATH = os.path.join(MODEL_DIR, "pose_landmarker_heavy.task")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("Manifest path:", MANIFEST_PATH)
print("Output folder:", OUTPUT_DIR)
print("Error log path:", ERROR_LOG_PATH)
print("MediaPipe model path:", MODEL_PATH)


# ============================================================
# 3. Download MediaPipe Pose Landmarker Model
# ============================================================

def download_pose_landmarker_model(model_path):
    """
    Download MediaPipe Pose Landmarker Heavy model if it does not exist.
    """
    if os.path.exists(model_path):
        print("MediaPipe model already exists:", model_path)
        return

    url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task"

    print("Downloading MediaPipe Pose Landmarker model...")
    urllib.request.urlretrieve(url, model_path)
    print("Downloaded:", model_path)


download_pose_landmarker_model(MODEL_PATH)


# ============================================================
# 4. MediaPipe Pose Setup
# ============================================================

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode


LANDMARK_NAMES = [
    "nose",
    "left_eye_inner", "left_eye", "left_eye_outer",
    "right_eye_inner", "right_eye", "right_eye_outer",
    "left_ear", "right_ear",
    "mouth_left", "mouth_right",
    "left_shoulder", "right_shoulder",
    "left_elbow", "right_elbow",
    "left_wrist", "right_wrist",
    "left_pinky", "right_pinky",
    "left_index", "right_index",
    "left_thumb", "right_thumb",
    "left_hip", "right_hip",
    "left_knee", "right_knee",
    "left_ankle", "right_ankle",
    "left_heel", "right_heel",
    "left_foot_index", "right_foot_index",
]


def make_columns():
    """
    Build CSV columns for one pose per frame.
    """
    cols = ["video_name", "video_path", "frame_idx", "timestamp_ms"]

    for name in LANDMARK_NAMES:
        cols += [
            f"{name}_x",
            f"{name}_y",
            f"{name}_z",
            f"{name}_visibility",
        ]

    return cols


CSV_COLUMNS = make_columns()

print("Number of landmark names:", len(LANDMARK_NAMES))
print("Number of CSV columns:", len(CSV_COLUMNS))


# ============================================================
# 5. Utility Functions for Pose Rows
# ============================================================

def empty_row(video_path, frame_idx, timestamp_ms):
    """
    Create an empty row with NaN keypoints.
    """
    row = {
        "video_name": os.path.basename(video_path),
        "video_path": video_path,
        "frame_idx": frame_idx,
        "timestamp_ms": timestamp_ms,
    }

    for name in LANDMARK_NAMES:
        row[f"{name}_x"] = np.nan
        row[f"{name}_y"] = np.nan
        row[f"{name}_z"] = np.nan
        row[f"{name}_visibility"] = np.nan

    return row


def fill_row_with_landmarks(row, landmarks):
    """
    Fill one row with MediaPipe landmarks.
    """
    for i, lm in enumerate(landmarks):
        if i >= len(LANDMARK_NAMES):
            break

        name = LANDMARK_NAMES[i]

        row[f"{name}_x"] = lm.x
        row[f"{name}_y"] = lm.y
        row[f"{name}_z"] = lm.z
        row[f"{name}_visibility"] = getattr(lm, "visibility", np.nan)

    return row


def get_video_fps(cap):
    """
    Safely get video FPS.
    """
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0 or np.isnan(fps):
        fps = 30.0

    return fps


# ============================================================
# 6. Single-Person Keypoint Extraction
# ============================================================

def extract_video_keypoints_to_df(video_path, model_path, frame_stride=3):
    """
    Extract keypoints using MediaPipe PoseLandmarker with num_poses=1.
    This is useful for simple testing.
    """
    rows = []

    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"Cannot open video: {video_path}")
        return pd.DataFrame(columns=CSV_COLUMNS)

    fps = get_video_fps(cap)

    frame_idx = 0

    with PoseLandmarker.create_from_options(options) as landmarker:
        while True:
            ret, frame = cap.read()

            if not ret:
                break

            if frame_idx % frame_stride != 0:
                frame_idx += 1
                continue

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb_frame,
            )

            timestamp_ms = int((frame_idx / fps) * 1000)
            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            row = empty_row(video_path, frame_idx, timestamp_ms)

            if result.pose_landmarks and len(result.pose_landmarks) > 0:
                landmarks = result.pose_landmarks[0]
                row = fill_row_with_landmarks(row, landmarks)

            rows.append(row)
            frame_idx += 1

    cap.release()

    return pd.DataFrame(rows, columns=CSV_COLUMNS)


# ============================================================
# 7. Main Actor Selection Helpers
# ============================================================

# MediaPipe landmark indices:
# 25, 26 = knees
# 27, 28 = ankles
# 31, 32 = foot_index
LEG_LANDMARK_IDX = [25, 26, 27, 28, 31, 32]


def get_pose_center_x(landmarks):
    """
    Estimate pose horizontal center using visible landmarks.
    """
    xs = [
        lm.x
        for lm in landmarks
        if getattr(lm, "visibility", 1.0) > 0.3
    ]

    if len(xs) == 0:
        return None

    return float(np.mean(xs))


def pose_to_array(landmarks):
    """
    Convert MediaPipe landmarks to numpy array:
    shape = [33, 4] -> x, y, z, visibility
    """
    if landmarks is None:
        return None

    arr = []

    for lm in landmarks:
        arr.append([
            lm.x,
            lm.y,
            lm.z,
            getattr(lm, "visibility", np.nan),
        ])

    return np.array(arr, dtype=float)


def compute_motion_score(track_arrays, visibility_threshold=0.3):
    """
    Compute a simple leg-motion score for one pose track.
    The main actor is assumed to have stronger leg movement.
    """
    score = 0.0
    prev = None

    for arr in track_arrays:
        if arr is None:
            continue

        if prev is not None:
            for idx in LEG_LANDMARK_IDX:
                x1, y1, _, v1 = prev[idx]
                x2, y2, _, v2 = arr[idx]

                valid = (
                    not np.isnan(x1) and not np.isnan(y1) and
                    not np.isnan(x2) and not np.isnan(y2) and
                    not np.isnan(v1) and not np.isnan(v2) and
                    v1 > visibility_threshold and
                    v2 > visibility_threshold
                )

                if valid:
                    score += np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

        prev = arr

    return score


# ============================================================
# 8. Main Actor Keypoint Extraction
# ============================================================

def extract_video_keypoints_main_actor(video_path, model_path, frame_stride=3, verbose=True):
    """
    Extract keypoints for the main actor.

    Logic:
    1. Detect up to 2 poses per sampled frame.
    2. Sort poses left-to-right for stable pose0 / pose1.
    3. Track leg motion score for both pose candidates.
    4. Choose the pose with higher leg motion as main actor.
    5. Save only the chosen actor's landmarks into CSV rows.
    """
    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.VIDEO,
        num_poses=2,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"Cannot open video: {video_path}")
        return pd.DataFrame(columns=CSV_COLUMNS)

    fps = get_video_fps(cap)

    frame_records = []
    track0 = []
    track1 = []

    frame_idx = 0

    with PoseLandmarker.create_from_options(options) as landmarker:
        while True:
            ret, frame = cap.read()

            if not ret:
                break

            if frame_idx % frame_stride != 0:
                frame_idx += 1
                continue

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb_frame,
            )

            timestamp_ms = int((frame_idx / fps) * 1000)
            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            poses = result.pose_landmarks if result.pose_landmarks else []

            # Sort candidates left-to-right by pose center.
            pose_pairs = []

            for landmarks in poses:
                cx = get_pose_center_x(landmarks)

                if cx is not None:
                    pose_pairs.append((cx, landmarks))

            pose_pairs.sort(key=lambda x: x[0])

            pose0 = pose_pairs[0][1] if len(pose_pairs) > 0 else None
            pose1 = pose_pairs[1][1] if len(pose_pairs) > 1 else None

            arr0 = pose_to_array(pose0)
            arr1 = pose_to_array(pose1)

            track0.append(arr0)
            track1.append(arr1)

            frame_records.append({
                "frame_idx": frame_idx,
                "timestamp_ms": timestamp_ms,
                "pose0": pose0,
                "pose1": pose1,
            })

            frame_idx += 1

    cap.release()

    score0 = compute_motion_score(track0)
    score1 = compute_motion_score(track1)

    chosen_key = "pose0" if score0 >= score1 else "pose1"

    if verbose:
        print(
            f"Selected {chosen_key}: "
            f"score0={score0:.4f}, score1={score1:.4f}"
        )

    rows = []

    for rec in frame_records:
        row = empty_row(
            video_path=video_path,
            frame_idx=rec["frame_idx"],
            timestamp_ms=rec["timestamp_ms"],
        )

        chosen_pose = rec[chosen_key]

        if chosen_pose is not None:
            row = fill_row_with_landmarks(row, chosen_pose)

        rows.append(row)

    return pd.DataFrame(rows, columns=CSV_COLUMNS)


# ============================================================
# 9. Pose Candidate Inspection Tool
# ============================================================

def inspect_two_pose_candidates(video_path, model_path, frame_number=30):
    """
    Display pose0 / pose1 candidates on a selected frame.
    Use this to check whether main actor selection is reasonable.
    """
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)

    ret, frame = cap.read()
    cap.release()

    if not ret:
        print("Cannot read frame:", frame_number)
        return

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.IMAGE,
        num_poses=2,
    )

    with PoseLandmarker.create_from_options(options) as landmarker:
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame,
        )

        result = landmarker.detect(mp_image)

    plt.figure(figsize=(10, 6))
    plt.imshow(rgb_frame)

    h, w, _ = rgb_frame.shape

    poses = result.pose_landmarks if result.pose_landmarks else []
    pose_pairs = []

    for landmarks in poses:
        cx = get_pose_center_x(landmarks)

        if cx is not None:
            pose_pairs.append((cx, landmarks))

    pose_pairs.sort(key=lambda x: x[0])

    for idx, (_, landmarks) in enumerate(pose_pairs):
        xs = [lm.x * w for lm in landmarks]
        ys = [lm.y * h for lm in landmarks]

        plt.scatter(xs, ys, s=10, label=f"pose{idx}")
        plt.text(
            np.mean(xs),
            np.mean(ys),
            f"pose{idx}",
            fontsize=14,
            color="yellow",
        )

    plt.title(f"Pose candidates at frame {frame_number}")
    plt.legend()
    plt.axis("off")
    plt.show()


# ============================================================
# 10. Load Clean Manifest
# ============================================================

if not os.path.exists(MANIFEST_PATH):
    raise FileNotFoundError(
        f"Clean manifest not found: {MANIFEST_PATH}\n"
        "Please run 01_data_processing_colab first."
    )

df_clean = pd.read_csv(MANIFEST_PATH)

print("Loaded clean manifest:", MANIFEST_PATH)
print("Total clean videos:", len(df_clean))

display(df_clean.head(10))


# ============================================================
# 11. Single Video Test
# ============================================================

def run_single_video_test(df_clean, test_index=0, frame_stride=3):
    """
    Test keypoint extraction on one video.
    """
    if len(df_clean) == 0:
        raise RuntimeError("df_clean is empty.")

    test_video = df_clean.iloc[test_index]["full_path"]

    print("Testing video:")
    print(test_video)

    if not os.path.exists(test_video):
        raise FileNotFoundError(test_video)

    df_test = extract_video_keypoints_main_actor(
        video_path=test_video,
        model_path=MODEL_PATH,
        frame_stride=frame_stride,
        verbose=True,
    )

    print("Extracted shape:", df_test.shape)
    display(df_test.head())

    test_csv_path = os.path.join(
        OUTPUT_DIR,
        os.path.splitext(os.path.basename(test_video))[0] + "_keypoints.csv",
    )

    df_test.to_csv(test_csv_path, index=False, encoding="utf-8-sig")

    print("Saved test CSV:", test_csv_path)

    return test_video, df_test, test_csv_path


# Change this index if you want to test another video.
# If you do not want to run a test, set RUN_SINGLE_TEST = False.
RUN_SINGLE_TEST = True

if RUN_SINGLE_TEST:
    test_video, df_test, test_csv_path = run_single_video_test(
        df_clean=df_clean,
        test_index=0,
        frame_stride=3,
    )


# ============================================================
# 12. Optional: Inspect Pose Candidates on Test Video
# ============================================================

# This helps you visually check pose0 / pose1.
# If the video has two people, inspect a few frames.
RUN_POSE_INSPECTION = True

if RUN_SINGLE_TEST and RUN_POSE_INSPECTION:
    for frame_number in [30, 60, 90]:
        print("Inspecting frame:", frame_number)
        inspect_two_pose_candidates(
            video_path=test_video,
            model_path=MODEL_PATH,
            frame_number=frame_number,
        )


# ============================================================
# 13. Batch Extraction Settings
# ============================================================

# Important:
# If you already generated many CSV files, keep FORCE_REPROCESS = False.
# This makes the batch job resume safely.

FORCE_REPROCESS = False

# Set START_INDEX if you want to resume from a later manifest row.
START_INDEX = 0

# Set BATCH_LIMIT for testing, for example 10.
# Set to None for full batch.
BATCH_LIMIT = None

# Frame stride:
# 3 means process one frame every 3 frames.
# Smaller = more detailed but slower.
# Larger = faster but may lose motion details.
FRAME_STRIDE = 3

# Files manually skipped due to known issues.
SKIP_FILES = {
    "IMG_6178",
}

print("FORCE_REPROCESS:", FORCE_REPROCESS)
print("START_INDEX:", START_INDEX)
print("BATCH_LIMIT:", BATCH_LIMIT)
print("FRAME_STRIDE:", FRAME_STRIDE)
print("SKIP_FILES:", SKIP_FILES)


# ============================================================
# 14. Batch Extract Keypoints
# ============================================================

def batch_extract_keypoints(
    df_clean,
    output_dir,
    model_path,
    error_log_path,
    frame_stride=3,
    start_index=0,
    batch_limit=None,
    skip_files=None,
    force_reprocess=False,
):
    """
    Batch process videos from clean manifest and save keypoint CSV files.

    Features:
    - Resume-safe: skips existing CSV files by default.
    - Error-safe: writes error log immediately when an error happens.
    - Main actor extraction: selects pose with stronger leg motion.
    """
    if skip_files is None:
        skip_files = set()

    os.makedirs(output_dir, exist_ok=True)

    if start_index < 0:
        start_index = 0

    df_run = df_clean.iloc[start_index:].copy()

    if batch_limit is not None:
        df_run = df_run.head(batch_limit).copy()

    print("Videos to check in this run:", len(df_run))
    print("Output folder:", output_dir)

    error_rows = []

    # Load existing error log if available.
    if os.path.exists(error_log_path):
        try:
            old_errors = pd.read_csv(error_log_path)
            error_rows = old_errors.to_dict("records")
            print("Loaded existing error log rows:", len(error_rows))
        except Exception:
            print("Could not read existing error log. Starting fresh.")

    processed_count = 0
    skipped_exists_count = 0
    skipped_manual_count = 0
    missing_count = 0
    error_count = 0

    for i, row in tqdm(df_run.iterrows(), total=len(df_run)):
        video_path = row["full_path"]
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        out_path = os.path.join(output_dir, f"{video_name}_keypoints.csv")

        try:
            if video_name in skip_files:
                print(f"[{i}] Skip manual: {video_name}")
                skipped_manual_count += 1
                continue

            if not os.path.exists(video_path):
                print(f"[{i}] Missing file: {video_name}")

                error_rows.append({
                    "index": i,
                    "video_name": video_name,
                    "video_path": video_path,
                    "error": "File does not exist",
                })

                pd.DataFrame(error_rows).to_csv(
                    error_log_path,
                    index=False,
                    encoding="utf-8-sig",
                )

                missing_count += 1
                continue

            if os.path.exists(out_path) and not force_reprocess:
                print(f"[{i}] Skip exists: {video_name}")
                skipped_exists_count += 1
                continue

            print(f"[{i}] Processing: {video_name}")
            print("Path:", video_path)

            df_kp = extract_video_keypoints_main_actor(
                video_path=video_path,
                model_path=model_path,
                frame_stride=frame_stride,
                verbose=True,
            )

            df_kp.to_csv(out_path, index=False, encoding="utf-8-sig")

            print(f"[{i}] Saved: {out_path} | shape={df_kp.shape}")

            processed_count += 1

        except Exception as e:
            print(f"[{i}] ERROR: {video_name}")
            print(e)

            error_rows.append({
                "index": i,
                "video_name": video_name,
                "video_path": video_path,
                "error": str(e),
                "traceback": traceback.format_exc(),
            })

            pd.DataFrame(error_rows).to_csv(
                error_log_path,
                index=False,
                encoding="utf-8-sig",
            )

            error_count += 1

    if error_rows:
        pd.DataFrame(error_rows).to_csv(
            error_log_path,
            index=False,
            encoding="utf-8-sig",
        )

        print("Error log saved:", error_log_path)
    else:
        print("Done. No errors.")

    summary = {
        "processed_count": processed_count,
        "skipped_exists_count": skipped_exists_count,
        "skipped_manual_count": skipped_manual_count,
        "missing_count": missing_count,
        "error_count": error_count,
        "output_dir": output_dir,
        "error_log_path": error_log_path,
    }

    print("\nBatch summary:")
    print(json.dumps(summary, indent=2))

    return summary


# ============================================================
# 15. Run Batch Extraction
# ============================================================

# IMPORTANT:
# Set RUN_BATCH_EXTRACTION = True when you are ready.
# If you only want to test functions, keep it False.

RUN_BATCH_EXTRACTION = False

if RUN_BATCH_EXTRACTION:
    batch_summary = batch_extract_keypoints(
        df_clean=df_clean,
        output_dir=OUTPUT_DIR,
        model_path=MODEL_PATH,
        error_log_path=ERROR_LOG_PATH,
        frame_stride=FRAME_STRIDE,
        start_index=START_INDEX,
        batch_limit=BATCH_LIMIT,
        skip_files=SKIP_FILES,
        force_reprocess=FORCE_REPROCESS,
    )
else:
    print("Batch extraction is not running.")
    print("To run it, set RUN_BATCH_EXTRACTION = True and rerun this section.")


# ============================================================
# 16. Output Folder Summary
# ============================================================

def summarize_output_csv_files(output_dir):
    """
    Count generated keypoint CSV files.
    """
    if not os.path.exists(output_dir):
        print("Output folder not found:", output_dir)
        return pd.DataFrame()

    rows = []

    for f in os.listdir(output_dir):
        if not f.endswith(".csv"):
            continue

        path = os.path.join(output_dir, f)

        rows.append({
            "csv_file": f,
            "csv_path": path,
            "video_name": f.replace("_keypoints.csv", ""),
            "size_mb": os.path.getsize(path) / 1024 / 1024,
        })

    df_summary = pd.DataFrame(rows)

    print("Generated CSV count:", len(df_summary))

    if len(df_summary) > 0:
        display(df_summary.head(20))

    return df_summary


df_output_summary = summarize_output_csv_files(OUTPUT_DIR)


# ============================================================
# 17. Final Output Summary
# ============================================================

print("\n============================================================")
print("02_keypoint_extraction_colab READY")
print("============================================================")
print("Input:")
print("1.", MANIFEST_PATH)
print("\nOutput:")
print("1.", OUTPUT_DIR)
print("2.", ERROR_LOG_PATH)
print("\nMain functions:")
print("- extract_video_keypoints_to_df()")
print("- extract_video_keypoints_main_actor()")
print("- inspect_two_pose_candidates()")
print("- batch_extract_keypoints()")
print("\nNext notebook:")
print("03_manual_labeling_colab")
print("============================================================")